# LangChain Tutorial
## Semantic search in documents using Vector Store Retriever with data sourced from PDFs
### Import Libraries

In [1]:
import json

from IPython.display import JSON, Markdown, display, display_json
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


/var/folders/j9/pgwv4z1143q_6zprltzrvf9w0000gn/T/ipykernel_22414/166081350.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
W1108 08:24:26.129000 22414 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### Create a few dumy documents

In [2]:
documents = [
    Document(
        page_content="AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.", 
        metadata={"source": "IT-LAPTOP-01"}),
    Document(
        page_content="AcmeOrbit travel policy: Expense claims must be submitted within 30 calendar days after travel.", 
        metadata={"source": "TRAVEL-02"}),
    Document(
        page_content="AcmeOrbit security policy: Employees must use multifactor authentication for company accounts.", 
        metadata={"source": "SECURITY-03"}),
]


### Load from PDF
PyPDFLoader is a class within the LangChain library used to load and parse PDF documents into text format. It is one of the most common document loaders for building LLM applications (like Retrieval-Augmented Generation or RAG) because it is simple to use and converts each page of a PDF into a standard LangChain Document object.


🔑 Key Features
* Page-by-Page Splitting: It automatically treats each page of the PDF as a separate Document.
* Metadata Tracking: Every extracted page includes metadata containing the source file path and the page number (starting at 0).
* Lazy Loading: It supports lazy_load(), which allows you to stream pages one by one to save memory when handling massive PDF files.

In [ ]:
file_path = "../data/llama_research_paper.pdf"
loader = PyPDFLoader(file_path=file_path)

docs = loader.load()
print(len(docs))  # The number of pages on the document

27


The PyPDFLoader from langchain_community, as we wrote above, does something of load PDF pages created over pypdf below:

In [19]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = "data/llama_research_paper.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

27



###  When not to use PyPDFLoader
PyPDFLoader relies on pure text extraction. You should avoid it or look for alternatives if:
1. Your PDF is scanned / an image: It does not perform OCR (Optical Character Recognition). You would need a loader that supports pytesseract or Amazon Textract instead.
2. Your PDF has complex tables: The cell alignment will likely break into a messy text string.

### Comparison with Alternatives

| Loader	| Pros	| Cons |
|---|---|---|
| PyPDFLoader	| Easy to set up, native to LangChain, keeps track of page numbers perfectly.	| Struggles with complex layouts, multi-column text, or images (OCR). |
| PyMuPDFLoader	| Extremely fast, extracts text in a cleaner reading order.	| Requires a different underlying C-library (fitz). |
| UnstructuredPDFLoader	| Great for complex layouts, tables, and unstructured formats.	| Slower, heavy on dependencies, harder to configure. |

See more alternatives at [Langchain web site](https://docs.langchain.com/oss/python/integrations/document_loaders#pdfs).

### Check the content of one document

In [4]:
n = 0
print("# Check one Document")
print("Content:\n", docs[n].page_content[:200])
print("-"*60)
print("Metadata:\n")
metadata = docs[n].metadata
for key in metadata:
    print(f"* {key}: {metadata[key]}")

# Check one Document
Content:
 LLaMA: Open and Efﬁcient Foundation Language Models
Hugo Touvron∗, Thibaut Lavril∗, Gautier Izacard∗, Xavier Martinet
Marie-Anne Lachaux, Timothee Lacroix, Baptiste Rozière, Naman Goyal
Eric Hambro, F
------------------------------------------------------------
Metadata:

* producer: pdfTeX-1.40.21
* creator: LaTeX with hyperref
* creationdate: 2023-02-28T01:57:46+00:00
* author: 
* keywords: 
* moddate: 2023-02-28T01:57:46+00:00
* ptex.fullbanner: This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2
* subject: 
* title: 
* trapped: /False
* source: data/llama_research_paper.pdf
* total_pages: 27
* page: 0
* page_label: 1


In [5]:
print(json.dumps(dict(docs[0]), indent=2))

{
  "id": null,
  "metadata": {
    "producer": "pdfTeX-1.40.21",
    "creator": "LaTeX with hyperref",
    "creationdate": "2023-02-28T01:57:46+00:00",
    "author": "",
    "keywords": "",
    "moddate": "2023-02-28T01:57:46+00:00",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "subject": "",
    "title": "",
    "trapped": "/False",
    "source": "data/llama_research_paper.pdf",
    "total_pages": 27,
    "page": 0,
    "page_label": "1"
  },
  "page_content": "LLaMA: Open and Ef\ufb01cient Foundation Language Models\nHugo Touvron\u2217, Thibaut Lavril\u2217, Gautier Izacard\u2217, Xavier Martinet\nMarie-Anne Lachaux, Timothee Lacroix, Baptiste Rozi\u00e8re, Naman Goyal\nEric Hambro, Faisal Azhar, Aurelien Rodriguez, Armand Joulin\nEdouard Grave\u2217, Guillaume Lample\u2217\nMeta AI\nAbstract\nWe introduce LLaMA, a collection of founda-\ntion language models ranging from 7B to 65B\nparameters. We train our models

### Update or Add Metadatas

In [6]:
description = """
    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. 
    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. 
    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best 
    large language models while only using publicly available data. The models and training methodology aim to improve access to large 
    language models and allow for open-sourcing of the models.
    """
for n in range(len(docs)):
    docs[n].metadata["title"] = "research paper Llama"
    docs[n].metadata["subject"] = "LLaMA: Open and Efficient Foundation Language Models"
    docs[n].metadata["description"] = description
    docs[n].metadata["source"] = "Meta AI"
    docs[n].metadata["type"] = "PDF"

### Split each documents into small chunck

In [10]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)
all_splits = text_spliter.split_documents(docs)

print(len(all_splits))
print(json.dumps(dict(all_splits[0]), indent=2))

118
{
  "id": null,
  "metadata": {
    "producer": "pdfTeX-1.40.21",
    "creator": "LaTeX with hyperref",
    "creationdate": "2023-02-28T01:57:46+00:00",
    "author": "",
    "keywords": "",
    "moddate": "2023-02-28T01:57:46+00:00",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "subject": "LLaMA: Open and Efficient Foundation Language Models",
    "title": "research paper Llama",
    "trapped": "/False",
    "source": "Meta AI",
    "total_pages": 27,
    "page": 0,
    "page_label": "1",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large 

### Creating and Store Embeddings 
#### Download HuggingFace models Once and Save/Load from a Local Directory
f you want to explicitly save the model files to a specific local folder (e.g., ./local_bge_small) and load directly from that path offline, download and save it via sentence-transformers or transformers.

In [7]:
local_model_path = "../models/local_bge_small"

# 1. Download and save to disk once
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
model.save(local_model_path)

# 2. Load HuggingFaceEmbeddings pointing to the local folder
embeddings = HuggingFaceEmbeddings(model_name=local_model_path)
vector_store = Chroma(embedding_function=embeddings)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Add the documents to the Vector Store

In [11]:
ids = vector_store.add_documents(documents=all_splits)
len(ids), ids[:5]

(118,
 ['8b65634a-df14-4374-9af1-f522de1e0c73',
  '80304132-e4e0-4962-8c33-e8001bd00894',
  '20180f60-10ff-44f2-b960-7474c1f9cd6e',
  '517f4293-eaee-424d-8753-f858030126d4',
  '4de70b06-3cfc-40cc-a108-3aeeccd62987'])

### Perform semantic Search

In [12]:
query = "What is the optimizer used?"
results = vector_store.similarity_search(query=query)
print("Total chunks:" ,len(results))
print("First Chunk:")
print("-"*60)
print(json.dumps(dict(results[0]), indent=2))


Total chunks: 4
First Chunk:
------------------------------------------------------------
{
  "id": "e7609259-286b-41a3-b672-c0e7e7d430f9",
  "metadata": {
    "source": "Meta AI",
    "title": "research paper Llama",
    "creator": "LaTeX with hyperref",
    "page_label": "3",
    "trapped": "/False",
    "page": 2,
    "keywords": "",
    "author": "",
    "producer": "pdfTeX-1.40.21",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large language models while only using publicly available data. The models and training methodology aim to improve access to large \n    language models and allow for open-sourcing of t

In [13]:
print(results[0].page_content[:250])

The details of the hyper-parameters for our dif-
ferent models are given in Table 2.
2.3 Optimizer
Our models are trained using the AdamW opti-
mizer (Loshchilov and Hutter, 2017), with the fol-
lowing hyper-parameters: β1 = 0.9,β 2 = 0.95.
We use a 


#### similarity_search_with_relevance_scores - Score based on Logic Similarity
This method abstracts away the mathematical calculations and focuses on the end-user experience. It assumes that you want the most relevant documents first, ordering the results in descending order of logical relevance (with the best match at the top).

In [14]:
results = vector_store.similarity_search_with_relevance_scores(query)
print("Total chunks:" ,len(results))
print("First Chunk:")
print("-"*60)
print("Score: ", results[0][1])
print(json.dumps(dict(results[0][0]), indent=2))

Total chunks: 4
First Chunk:
------------------------------------------------------------
Score:  0.5853665536077683
{
  "id": "e7609259-286b-41a3-b672-c0e7e7d430f9",
  "metadata": {
    "start_index": 1567,
    "creationdate": "2023-02-28T01:57:46+00:00",
    "creator": "LaTeX with hyperref",
    "author": "",
    "trapped": "/False",
    "type": "PDF",
    "moddate": "2023-02-28T01:57:46+00:00",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "keywords": "",
    "page": 2,
    "page_label": "3",
    "producer": "pdfTeX-1.40.21",
    "subject": "LLaMA: Open and Efficient Foundation Language Models",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms G

In [15]:
print("Scores by Chunks:")
for num, chunk in enumerate(results, start=1):
    print(f"Score of Chunk {num}: ", chunk[1])

Scores by Chunks:
Score of Chunk 1:  0.5853665536077683
Score of Chunk 2:  0.5600301442128386
Score of Chunk 3:  0.5539636954255744
Score of Chunk 4:  0.5117480211110621


#### similarity_search_by_vector_with_relevance_scores - Score based on Distance
This method exposes the raw data calculated by the underlying vector database (such as Chroma, FAISS, Pinecone, etc.). If your vector database is configured to use L2 (Euclidean) distance or Cosine distance (rather than Cosine similarity), a lower number indicates that the vectors are closer together in space. Therefore, the lowest score represents the greatest proximity (the best result). It lists the results from the smallest error/distance to the largest.

In [16]:
embedding = embeddings.embed_query(query)
results2 = vector_store.similarity_search_by_vector_with_relevance_scores(embedding=embedding)

print("Scores by Chunks:")
for num, chunk in enumerate(results, start=1):
    print(f"Score of Chunk {num}: ", chunk[1])

Scores by Chunks:
Score of Chunk 1:  0.5853665536077683
Score of Chunk 2:  0.5600301442128386
Score of Chunk 3:  0.5539636954255744
Score of Chunk 4:  0.5117480211110621


In [58]:
results[0]

(Document(id='e27e63bb-7ea6-4c7a-9df8-72f6f45c86cf', metadata={'start_index': 1567, 'creator': 'LaTeX with hyperref', 'title': 'research paper Llama', 'total_pages': 27, 'description': '\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large language models while only using publicly available data. The models and training methodology aim to improve access to large \n    language models and allow for open-sourcing of the models.\n    ', 'author': '', 'page': 2, 'creationdate': '2023-02-28T01:57:46+00:00', 'source': 'Meta AI', 'page_label': '3', 'moddate': '2023-02-28T01:57:46+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX,

### Set a retriever to configure the serach and retriever process, like to get top n similar searchs

In [17]:
retriever = vector_store.as_retriever(search_type="similarity_score_threshold", search_kwargs={ "k": 3, "score_threshold": 0.5})
results = retriever.invoke(query)
print(len(results))

3


#### If you need this inside a Chain (LCEL)
If you are using this in a chain and absolutely need the scores passed to the next step, you can create a simple custom function:

In [18]:
def retrieve_with_scores(query):
    # Call the vector store directly inside the function
    return vector_store.similarity_search_with_relevance_scores(query, score_threshold=0.5, k=3)

def print_scores(results):
    print("Scores by Chunks:")
    for num, chunk in enumerate(results, start=1):
        print(f"Score of Chunk {num}: ", chunk[1])

# Wrap it in a RunnableLambda to use it in a chain
chain = RunnableLambda(retrieve_with_scores) | print_scores
chain.invoke(query)

Scores by Chunks:
Score of Chunk 1:  0.5853665536077683
Score of Chunk 2:  0.5600301442128386
Score of Chunk 3:  0.5539636954255744
